In [1]:
from pathlib import Path
import pandas as pd

cwd = Path.cwd().resolve()
project_root = next(p for p in [cwd, *cwd.parents] if (p / "annotations" / "combined_clean.csv").exists())

csv_path = project_root / "annotations" / "combined_clean.csv"

print("CWD:", cwd)
print("Project root:", project_root)
print("CSV exists:", csv_path, "exists:", csv_path.exists())


df = pd.read_csv(csv_path)
print("rader:", len(df))


CWD: /home/andre-kidess/studier/DAT191/visual-vehicle-recognition-varying-lighting-conditions/notebooks
Project root: /home/andre-kidess/studier/DAT191/visual-vehicle-recognition-varying-lighting-conditions
CSV exists: /home/andre-kidess/studier/DAT191/visual-vehicle-recognition-varying-lighting-conditions/annotations/combined_clean.csv exists: True
rader: 5380


In [2]:
import numpy as np
from sklearn.model_selection import train_test_split

SEED = 42
TEST_SIZE = 0.15
VAL_SIZE_TOTAL = 0.15
VAL_SIZE_REL = VAL_SIZE_TOTAL / (1.0 - TEST_SIZE)  # val-andel av trainval

# Anta df har minst: ["model", "lighting"] (+ image/color etc.)
df["model"] = df["model"].astype("string")
df["lighting"] = df["lighting"].astype("string")


model_norm = df["model"].astype(str).str.strip().str.lower()
# Nivå 1: Tesla vs Other (robust mot case/whitespace)
# Treffer "other", "other car", "other_car", osv.
is_other = model_norm.str.startswith("other")
df["lvl1"] = np.where(is_other, "Other", "Tesla")

# lvl2 skal kun brukes for Tesla – for Other settes NA
df["lvl2"] = np.where(df["lvl1"] == "Tesla", df["model"].astype(str), "NA")

print(df["lvl1"].value_counts(dropna=False))
print(df.loc[df["lvl1"]=="Other", "model"].value_counts().head(10))
print(df.loc[df["lvl1"]=="Tesla", "model"].value_counts().head(10))

# Stratify-key: nivå1 x nivå2 x lyskategori
df["strat_key"] = df["lvl1"] + "|" + df["lvl2"] + "|" + df["lighting"]

# (Valgfritt, men nyttig) sjekk små strata før split
counts = df["strat_key"].value_counts()
rare = counts[counts < 3]  # terskel kan justeres
if len(rare):
    print("⚠️ Strata med veldig få observasjoner (kan gi stratify-problemer):")
    print(rare)

# 1) trainval vs test
trainval_df, test_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=df["strat_key"],
)

# 2) train vs val (på trainval)
train_df, val_df = train_test_split(
    trainval_df,
    test_size=VAL_SIZE_REL,
    random_state=SEED,
    stratify=trainval_df["strat_key"],
)

print(len(train_df), len(val_df), len(test_df))

lvl1
Other    3099
Tesla    2281
Name: count, dtype: int64
model
Other car    3099
Name: count, dtype: Int64
model
Y 2020–2024    856
3 2017–2023    344
Y 2025-nå      326
X              267
3 2024–nå      246
S 2016–nå      134
S 2012–2015    108
Name: count, dtype: Int64
3766 807 807


In [3]:


def stratified_split_light_table(train_df, val_df, test_df,
                                 lighting_col="lighting",
                                 lighting_order=None,
                                 split_order=("train", "val", "test")):
    parts = {"train": train_df.copy(), "val": val_df.copy(), "test": test_df.copy()}

    # --- lysrekkefølge ---
    all_lights = pd.concat(
        [parts[s][lighting_col].astype("string").str.strip() for s in split_order],
        ignore_index=True
    ).dropna()

    if lighting_order is None:
        preferred = ["lyst", "lavlys", "mørkt"]
        uniq = [x for x in preferred if x in set(all_lights.unique())]
        rest = sorted(set(all_lights.unique()) - set(uniq))
        lighting_order = uniq + rest

    # --- bygg long-format (class_label, split, lighting, n) ---
    rows = []
    for split in split_order:
        d = parts[split]

        # Radlabel: Tesla -> lvl2, Other -> "Other"
        class_label = d["lvl2"].astype("string").copy()
        class_label[d["lvl1"].eq("Other")] = "Other"

        lighting = d[lighting_col].astype("string").str.strip()

        g = (
            pd.DataFrame({"class_label": class_label, "lighting": lighting})
            .groupby(["class_label", "lighting"])
            .size()
            .reset_index(name="n")
        )
        g["split"] = split
        rows.append(g)

    long_df = pd.concat(rows, ignore_index=True)

    # --- pivot til wide ---
    wide = long_df.pivot_table(
        index="class_label",
        columns=["split", "lighting"],
        values="n",
        aggfunc="sum",
        fill_value=0
    )

    # Sørg for alle kolonner finnes (split x lighting)
    wide = wide.reindex(
        columns=pd.MultiIndex.from_product([list(split_order), list(lighting_order)]),
        fill_value=0
    )

    # Nå skal alt være tall uten NaN
    wide = wide.fillna(0).astype(int)

    # --- totals (legg til etter cast, så du unngår NaN-problemer) ---
    wide[("TOTAL", "ALL")] = wide.sum(axis=1).astype(int)
    col_totals = wide.sum(axis=0).astype(int)
    wide.loc["TOTAL"] = col_totals

    return wide

# ---- usage ----
table = stratified_split_light_table(train_df, val_df, test_df)
table

train               val              test              TOTAL
             Dark Light Medium Dark Light Medium Dark Light Medium   ALL
class_label                                                             
3 2017–2023    72   100     69   15    21     15   16    21     15   344
3 2024–nå      55    63     54   12    13     12   12    13     12   246
Other         421  1313    436   90   282     93   90   281     93  3099
S 2012–2015     7    62      7    1    13      2    1    13      2   108
S 2016–nå      11    72     11    2    15      3    2    15      3   134
X              30   139     17    7    30      4    6    30      4   267
Y 2020–2024   164   273    163   35    58     35   35    58     35   856
Y 2025-nå      86    46     95   18    10     21   19    10     21   326
TOTAL         846  2068    852  180   442    185  181   441    185  5380

In [4]:
from pathlib import Path

project_root =Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
    
out_dir = project_root / "datasplitt"
out_dir.mkdir(parents=True, exist_ok=True)

train_path = out_dir / "train.csv"
val_path   = out_dir / "val.csv"
test_path  = out_dir / "test.csv"

train_df.to_csv(train_path, index=False, encoding="utf-8")
val_df.to_csv(val_path, index=False, encoding="utf-8")
test_df.to_csv(test_path, index=False, encoding="utf-8")

print("Saved:")
print(" -", train_path.resolve())
print(" -", val_path.resolve())
print(" -", test_path.resolve())

Saved:
 - /home/andre-kidess/studier/DAT191/visual-vehicle-recognition-varying-lighting-conditions/datasplitt/train.csv
 - /home/andre-kidess/studier/DAT191/visual-vehicle-recognition-varying-lighting-conditions/datasplitt/val.csv
 - /home/andre-kidess/studier/DAT191/visual-vehicle-recognition-varying-lighting-conditions/datasplitt/test.csv
